# FACI: bounded NLI reproduction

This review copy runs the corrected BERT/RoBERTa NLI measurements and their
pretrained-feature comparisons. It uses the original measurement, calibration,
aggregation, and stopping code. Packaging changes load the included source tree
and frozen input manifest; historical saved-result comparisons are omitted.
It does not claim to reproduce every historical result in the paper.

## Run in Colab

1. Open this notebook in Colab and select a GPU runtime. The recorded run used
   a Tesla T4 with Python 3.13.15, PyTorch 2.11.0+cu128, and CUDA 12.8. Dependency
   versions installed below match the recorded run; Colab supplies PyTorch.
2. Run **Cell 1** once, restart the runtime, and then continue from **Cell 2**.
3. Authorize your own Google Drive. Outputs are written to the separate folder
   `MyDrive/FACI_NLI_Review`. Use a Drive without that folder for a fresh run.
4. When Cell 2 requests a file, upload the entire lean review repository ZIP.
   It must contain `historical/code/` and `inputs/nli_manifest.json` under one
   repository directory. A GitHub Download ZIP with the same layout also works.
5. Run Cells 3–10 sequentially. Cells 7–9 perform calibration and model runs.
   Cell 10 summarizes and downloads `FACI_NLI_Review_results.zip`.

Internet access is required for the pinned public checkpoints.
The input manifest contains MNLI examples and revisions, not measurement
outputs. It is frozen before the unchanged protocol cell executes, so that
cell reuses the pinned inputs instead of resolving current repository heads.
The protocol's `source_zip_sha256` identifies the uploaded review ZIP.

Existing outputs in `FACI_NLI_Review` are checked and resumed. Frozen input,
environment, or code mismatches stop execution; use a separate fresh output
folder for a different run. No completed original experiment folder is used.
Numerical agreement can depend on the GPU/software stack; a fresh run is not
promised to be bitwise identical across platforms.


## Cell 1. Install dependencies

Restart the runtime after this cell.

In [ ]:
%pip install --quiet --only-binary=tokenizers \
    "transformers==4.51.3" \
    "tokenizers==0.21.4" \
    "huggingface-hub==0.34.4" \
    "datasets==3.6.0" \
    "numpy==2.1.3" \
    "pandas==2.2.3" \
    "scipy==1.15.3" \
    "scikit-learn==1.6.1" \
    "omegaconf==2.3.0" \
    "PyYAML==6.0.2" \
    "tqdm==4.67.1" \
    "threadpoolctl==3.6.0"

## Cell 2. Load review source and frozen inputs

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys, types, json, hashlib, zipfile, tempfile, random, gc, time
import inspect, platform, warnings, importlib.machinery
from pathlib import Path

import numpy as np
import pandas as pd
import torch, transformers, datasets, sklearn, scipy
import tokenizers, huggingface_hub

from google.colab import drive, files
from huggingface_hub import HfApi
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification
)
from datasets import load_dataset
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "Select a GPU runtime."

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)
torch.use_deterministic_algorithms(True)

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/FACI_NLI_Review")
ROOT.mkdir(parents=True, exist_ok=True)


def clean_json(x):
    if isinstance(x, dict):
        return {str(k): clean_json(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [clean_json(v) for v in x]
    if isinstance(x, np.ndarray):
        return clean_json(x.tolist())
    if isinstance(x, np.generic):
        return clean_json(x.item())
    if isinstance(x, float) and not np.isfinite(x):
        return None
    return x


def save(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(clean_json(value), indent=2, allow_nan=False)
    )
    tmp.replace(path)


def freeze(path, value):
    value = clean_json(value)
    if path.exists():
        assert json.loads(path.read_text()) == value, (
            f"Frozen setting changed: {path}"
        )
    else:
        save(path, value)


def remember_cell(name):
    freeze(
        ROOT / "notebook_cells" / f"{name}.json",
        {"source": get_ipython().history_manager.input_hist_raw[-1]}
    )


ENV = dict(
    python=platform.python_version(),
    torch=str(torch.__version__),
    transformers=transformers.__version__,
    datasets=datasets.__version__,
    numpy=np.__version__,
    sklearn=sklearn.__version__,
    scipy=scipy.__version__,
    tokenizers=tokenizers.__version__,
    huggingface_hub=huggingface_hub.__version__,
    cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0)
)
freeze(ROOT / "environment.json", ENV)
print(ENV)

# Upload the lean review repository ZIP. It supplies source code and
# frozen input selection, not completed model measurements.
from io import BytesIO

archive = ROOT / "source_review.zip"

if not archive.exists():
    uploaded = files.upload()
    assert len(uploaded) == 1, "Upload only the lean FACI review repository ZIP."
    data = next(iter(uploaded.values()))
else:
    data = archive.read_bytes()

ZIP_SHA = hashlib.sha256(data).hexdigest()
EXTRACT = Path(tempfile.mkdtemp(prefix="faci_nli_review_", dir="/content"))

with zipfile.ZipFile(BytesIO(data)) as z:
    for member in z.infolist():
        path = Path(member.filename)
        assert not path.is_absolute() and ".." not in path.parts, "Unsafe ZIP path."
        assert (EXTRACT / path).resolve().is_relative_to(EXTRACT), "Unsafe ZIP path."
        # Do not accept Unix symlink entries from uploaded archives.
        assert ((member.external_attr >> 16) & 0o170000) != 0o120000, "ZIP symlink rejected."
    z.extractall(EXTRACT)

matches = list(EXTRACT.rglob("historical/code/src/mechinterp/tiers/tier3.py"))
assert len(matches) == 1, "Expected one historical/code source tree in the review ZIP."
CODE = matches[0].parents[3]
PACKAGE = CODE.parent
REPOSITORY = PACKAGE.parent
INPUT_MANIFEST = REPOSITORY / "inputs/nli_manifest.json"
assert INPUT_MANIFEST.is_file(), "The review ZIP must include inputs/nli_manifest.json."
INPUT_MANIFEST_SHA256 = hashlib.sha256(INPUT_MANIFEST.read_bytes()).hexdigest()
assert INPUT_MANIFEST_SHA256 == "cdb96e3d16753b7bcf4856a404b82ecaa4c35e3923a51d60eb7d147f8c66a421", (
    "The frozen NLI input manifest differs from the supplied experiment inputs."
)

# This manifest records checkpoint revisions and dataset/example selection.
# Freezing it here makes the unchanged protocol cell use the recorded inputs.
input_manifest = json.loads(INPUT_MANIFEST.read_text(encoding="utf-8"))
assert set(input_manifest) == {
    "revisions", "dataset", "split", "dataset_revision",
    "dataset_fingerprint", "samples", "auxiliary"
}, "Unexpected input manifest fields."
freeze(ROOT / "manifest.json", input_manifest)
freeze(ROOT / "review_input.json", {
    "archive_sha256": ZIP_SHA,
    "manifest_sha256": INPUT_MANIFEST_SHA256,
})

if not archive.exists():
    archive.write_bytes(data)


def source_hashes():
    return {
        str(p.relative_to(CODE)):
        hashlib.sha256(p.read_bytes()).hexdigest()
        for p in sorted(CODE.rglob("*.py"))
    }


SOURCE_HASHES = source_hashes()
freeze(ROOT / "source_hashes.json", SOURCE_HASHES)

# The package initializer imports an absent baselines module.
# Import its existing submodules without executing that initializer.
assert "mechinterp" not in sys.modules, (
    "Restart the session before rerunning Cell 2."
)

sys.dont_write_bytecode = True
sys.path[:0] = [
    str(CODE / "src"),
    str(CODE / "scripts")
]

pkg = types.ModuleType("mechinterp")
pkg.__package__ = "mechinterp"
pkg.__path__ = [str(CODE / "src/mechinterp")]
pkg.__spec__ = importlib.machinery.ModuleSpec(
    "mechinterp", loader=None, is_package=True
)
pkg.__spec__.submodule_search_locations = pkg.__path__
sys.modules["mechinterp"] = pkg

from mechinterp.architectures.encoder import create_encoder_wrapper
from mechinterp.architectures.base import HookedModelWrapper
from mechinterp.tasks.nli import NLITask
from mechinterp.tiers.tier1 import Tier1Analyzer
from mechinterp.tiers.tier2 import Tier2Analyzer
from mechinterp.tiers.tier3 import Tier3Analyzer
from tier1_heldout_analyze import heldout_tier1
import tier2_heldout_analyze as t2module

# Preserve Tier 2's calculation, but raise errors instead of
# silently substituting zero when an inner-CV fit fails.
t2_source = inspect.getsource(t2module.heldout_tier2)
assert t2_source.count("except Exception: a=0.0") == 1

t2_source = t2_source.replace(
    "except Exception: a=0.0",
    "except Exception: raise"
)
t2_namespace = dict(t2module.__dict__)


def strict_cv(*args, **kwargs):
    kwargs["error_score"] = "raise"
    return t2module.cross_val_score(*args, **kwargs)


t2_namespace["cross_val_score"] = strict_cv
exec(
    compile(t2_source, "<strict_original_tier2>", "exec"),
    t2_namespace
)
heldout_tier2 = t2_namespace["heldout_tier2"]

freeze(ROOT / "strict_tier2.json", {"source": t2_source})
remember_cell("02_setup")

print("Original source imported; no source files edited.")

## Cell 3. Freeze the experimental protocol

In [ ]:
SEEDS = [42, 123, 456, 789, 1000, 2024, 3141, 5555, 7777, 9999]

SPECS = {
    "bert_nli": dict(
        checkpoint="textattack/bert-base-uncased-MNLI",
        task="classification",
        ent=1, con=0, neutral=2
    ),
    "roberta_nli": dict(
        checkpoint="textattack/roberta-base-MNLI",
        task="classification",
        ent=2, con=0, neutral=1
    ),
    "bert_pretrained": dict(
        checkpoint="bert-base-uncased",
        task="features"
    ),
    "roberta_pretrained": dict(
        checkpoint="roberta-base",
        task="features"
    ),
}

PROTOCOL = dict(
    version=1,
    seeds=SEEDS,
    n_positive=100,
    n_negative=100,
    n_patch_pairs=10,
    n_permutations=200,
    gamma=0.01,
    alpha=0.5,
    minimum_valid_pairs=3,
    threshold_operator=">=; exact zero excluded",
    gap_thresholds=[0.0, 0.01, 0.05, 0.10],
    auxiliary_pairs=20,
    dtype="float32",
    attention="sdpa/math",
    max_length=512,
    layer_statistic="median across retained pairs",
    C="mean across seeds of median across layers of pair medians",
    d50="first >=0.5 crossing of median-across-seeds pair-median profile",
    source_zip_sha256=ZIP_SHA,
    models=SPECS
)
freeze(ROOT / "protocol.json", PROTOCOL)

mp = ROOT / "manifest.json"

if mp.exists():
    MANIFEST = json.loads(mp.read_text())

else:
    api = HfApi()

    revisions = {
        k: api.model_info(
            v["checkpoint"],
            revision=(
                "3a97f689528cbd91bcc71ab29ea6c20c089d8f28"
                if k == "bert_nli" else "main"
            )
        ).sha
        for k, v in SPECS.items()
    }

    ds_revision = api.dataset_info("nyu-mll/glue").sha

    ds = load_dataset(
        "nyu-mll/glue",
        "mnli",
        split="validation_matched",
        revision=ds_revision
    )

    rows = [
        dict(
            source_row=i,
            idx=int(r["idx"]),
            premise=r["premise"],
            hypothesis=r["hypothesis"],
            label=int(r["label"])
        )
        for i, r in enumerate(ds)
        if r["label"] in (0, 2)
    ]

    task = NLITask()
    task._data = ds.filter(lambda r: r["label"] in (0, 2))

    def sample(label, n, seed):
        selected = [r for r in rows if r["label"] == label]
        if seed is not None:
            random.Random(seed).shuffle(selected)
        return selected[:n]

    def text_of(r):
        return r["premise"] + " [SEP] " + r["hypothesis"]

    samples = {}

    for seed in SEEDS:
        p = sample(0, 100, seed)
        n = sample(2, 100, seed)

        # Verify exact agreement with the original task sampler.
        assert [text_of(r) for r in p] == (
            task.get_positive_examples(100, seed=seed)
        )
        assert [text_of(r) for r in n] == (
            task.get_negative_examples(100, seed=seed)
        )

        samples[str(seed)] = dict(positive=p, negative=n)

    # Original auxiliary scripts used unshuffled positive examples
    # and seed-42 shuffled negative examples.
    auxiliary = dict(
        positive=sample(0, 20, None),
        negative=sample(2, 20, 42)
    )

    assert [text_of(r) for r in auxiliary["positive"]] == (
        task.get_positive_examples(40)[:20]
    )
    assert [text_of(r) for r in auxiliary["negative"]] == (
        task.get_negative_examples(40, seed=42)[:20]
    )

    MANIFEST = dict(
        revisions=revisions,
        dataset="nyu-mll/glue",
        split="validation_matched",
        dataset_revision=ds_revision,
        dataset_fingerprint=ds._fingerprint,
        samples=samples,
        auxiliary=auxiliary
    )
    save(mp, MANIFEST)


def text_of(r):
    return r["premise"] + " [SEP] " + r["hypothesis"]


remember_cell("03_protocol")
print("Frozen checkpoint revisions:", MANIFEST["revisions"])

## Cell 4. Define model interfaces and recovery calculations

In [ ]:
def paired_tokenize(self, text, **kwargs):
    texts = [text] if isinstance(text, str) else list(text)

    assert texts and all(
        isinstance(t, str) and t.count(" [SEP] ") == 1
        for t in texts
    ), "Missing or ambiguous NLI separator."

    pairs = [t.split(" [SEP] ") for t in texts]

    options = dict(
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    options.update(kwargs)

    tokens = self.tokenizer(
        [p[0] for p in pairs],
        text_pair=[p[1] for p in pairs],
        **options
    )
    return {k: v.to(self.device) for k, v in tokens.items()}


def load_wrapper(key):
    s = SPECS[key]
    rev = MANIFEST["revisions"][key]

    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)

    w = create_encoder_wrapper(
        s["checkpoint"], device="cuda", task=s["task"]
    )

    # Load explicitly: the original wrapper's load() does not
    # forward revision arguments.
    w._tokenizer = AutoTokenizer.from_pretrained(
        s["checkpoint"], revision=rev
    )
    cls = (
        AutoModelForSequenceClassification
        if s["task"] == "classification"
        else AutoModel
    )
    w._model = cls.from_pretrained(
        s["checkpoint"],
        revision=rev,
        torch_dtype=torch.float32,
        attn_implementation="sdpa"
    ).to("cuda").eval()

    w._tokenize = types.MethodType(paired_tokenize, w)
    return w


def tokens_cpu(w, text):
    return {
        k: v.detach().cpu().tolist()
        for k, v in w._tokenize(text).items()
    }


def collect_pair(w, donor, recipient):
    a = Tier3Analyzer(w, min_gap=0.01)
    captured = []

    def record(logits):
        assert tuple(logits.shape) == (1, 3)
        captured.append(
            logits[0].detach().float().cpu().tolist()
        )
        # This callback only records logits.
        # Recovery is calculated afterward.
        return 0.0

    try:
        x, cache = w.forward(donor)
        y, _ = w.forward(recipient)

        assert all(
            f"layer_{l}" in cache.hidden_states
            for l in range(w.num_layers)
        )

        for l in range(w.num_layers):
            a._run_with_patch(
                recipient,
                l,
                cache.hidden_states[f"layer_{l}"],
                output_fn=record
            )

        raw = dict(
            donor_logits=x[0].cpu().tolist(),
            recipient_logits=y[0].cpu().tolist(),
            patched_logits=captured,
            donor_tokens=tokens_cpu(w, donor),
            recipient_tokens=tokens_cpu(w, recipient)
        )

        assert np.isfinite([
            *raw["donor_logits"],
            *raw["recipient_logits"],
            *np.ravel(captured)
        ]).all(), "Nonfinite model output."

        np.testing.assert_allclose(
            captured[-1],
            raw["donor_logits"],
            rtol=1e-5,
            atol=1e-5
        )

        raw["max_final_layer_logit_error"] = float(
            np.max(
                np.abs(
                    np.array(captured[-1]) - raw["donor_logits"]
                )
            )
        )
        return raw

    finally:
        a._clear_hooks()
        w._clear_hooks()
        w._clear_cache()


def recoveries(raws, ent, con, gamma):
    donor = np.array(
        [r["donor_logits"] for r in raws], dtype=np.float32
    )
    recipient = np.array(
        [r["recipient_logits"] for r in raws], dtype=np.float32
    )
    patch = np.array(
        [r["patched_logits"] for r in raws], dtype=np.float32
    )

    # Preserve original arithmetic:
    # FP32 logit subtraction, then Python-float normalization.
    a = (donor[:, ent] - donor[:, con]).astype(np.float64)
    b = (recipient[:, ent] - recipient[:, con]).astype(np.float64)
    p = (patch[:, :, ent] - patch[:, :, con]).astype(np.float64)

    d = a - b
    numerator = p - b[:, None]

    keep = np.isfinite(d) & (np.abs(d) >= gamma) & (d != 0)

    R = np.full(numerator.shape, np.nan)
    R[keep] = numerator[keep] / d[keep, None]

    return d, keep, R


def summarize(raws, ent, con, gamma=0.01):
    d, keep, R = recoveries(raws, ent, con, gamma)

    n = int(keep.sum())
    vpr = float(keep.mean())
    enough = n >= 3

    med = (
        np.median(R[keep], axis=0)
        if enough else np.full(R.shape[1], np.nan)
    )
    mean = (
        np.mean(R[keep], axis=0)
        if enough else np.full(R.shape[1], np.nan)
    )
    crossed = np.flatnonzero(med >= 0.5)

    return clean_json(dict(
        gamma=gamma,
        n_pairs=len(raws),
        n_valid=n,
        VPR=vpr,
        sufficient_for_statistics=enough,
        applicable=enough and vpr >= 0.5,
        C=float(np.median(med)),
        layer_medians=med,
        layer_means=mean,
        d50_seed=(
            float(crossed[0] / (len(med) - 1))
            if len(crossed) else None
        ),
        mean_abs_gap=float(np.abs(d).mean()),
        gaps=d
    ))


def get_raws(w, folder, positive, negative, n):
    result = []

    for i in tqdm(range(n), desc=folder.name):
        p = folder / f"pair_{i:03d}.json"
        examples = dict(
            donor=positive[i],
            recipient=negative[i]
        )

        if p.exists():
            r = json.loads(p.read_text())
            assert r["examples"] == examples

        else:
            r = collect_pair(
                w,
                text_of(positive[i]),
                text_of(negative[i])
            )
            r["examples"] = examples
            save(p, r)

        result.append(r)

    return result


def clear_model(w):
    w._clear_hooks()
    w._clear_cache()
    w._model = None
    w._tokenizer = None
    gc.collect()
    torch.cuda.empty_cache()


remember_cell("04_measurement")

## Cell 5. Define calibration and implementation checks

In [ ]:
def check_model(key, w):
    s = SPECS[key]
    samples = MANIFEST["samples"]["42"]

    donor = text_of(samples["positive"][0])
    recipient = text_of(samples["negative"][0])
    p, h = donor.split(" [SEP] ")

    expected = w.tokenizer(
        [p],
        text_pair=[h],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    got = w._tokenize(donor)

    assert set(expected) == set(got)
    for k in expected:
        assert torch.equal(expected[k], got[k].cpu())

    assert tokens_cpu(w, donor) == tokens_cpu(w, [donor])

    if key.startswith("bert"):
        assert 1 in got["token_type_ids"][0].tolist()

    if s["task"] != "classification":
        return {"paired_tokenization": "passed"}

    raw = collect_pair(w, donor, recipient)
    own = collect_pair(w, donor, donor)

    np.testing.assert_allclose(
        own["patched_logits"],
        np.tile(own["donor_logits"], (w.num_layers, 1)),
        rtol=1e-5,
        atol=1e-5
    )

    fn = lambda z: float(
        (z[0, s["ent"]] - z[0, s["con"]]).item()
    )

    a = Tier3Analyzer(w, min_gap=0.01)
    try:
        layer = w.num_layers // 2

        direct = a.patch_layer(
            donor, recipient, layer, output_fn=fn
        )

        v = raw["patched_logits"][layer]
        np.testing.assert_allclose(
            v[s["ent"]] - v[s["con"]],
            direct["patched_output"],
            rtol=1e-5,
            atol=1e-5
        )
    finally:
        a._clear_hooks()
        w._clear_hooks()
        w._clear_cache()

    return dict(
        paired_tokenization="passed",
        original_patch_agreement="passed",
        max_self_patch_logit_error=float(
            np.max(
                np.abs(
                    np.array(own["patched_logits"])
                    - own["donor_logits"]
                )
            )
        ),
        config_id2label=w.model.config.id2label,
        fixed_label_indices={
            k: s[k] for k in ("ent", "con", "neutral")
        }
    )


def calibration(key):
    w = load_wrapper(key)
    s = SPECS[key]
    samples = MANIFEST["samples"]["42"]

    try:
        checks = check_model(key, w)
        save(ROOT / "checks" / f"{key}.json", checks)

        label_checks = []
        examples = [
            ("A person is eating.", "ent"),
            ("No one is eating.", "con"),
            ("The apple was grown in Italy.", "neutral"),
        ]

        for hypothesis, label in examples:
            text = "A person is eating an apple. [SEP] " + hypothesis

            with torch.no_grad():
                logits = w.model(
                    **w._tokenize(text)
                ).logits[0].cpu()

            label_checks.append(dict(
                hypothesis=hypothesis,
                expected_index=s[label],
                predicted_index=int(logits.argmax()),
                logits=logits.tolist()
            ))

        save(
            ROOT / "checks" / f"{key}_label_examples.json",
            label_checks
        )
        display(pd.DataFrame(label_checks))

        assert all(
            r["expected_index"] == r["predicted_index"]
            for r in label_checks
        ), (
            "Label sanity check disagreed. This may be a model error, "
            "not a label error. Send this output; do not change "
            "mappings to fit it."
        )

        rows = []

        for mode in ("legacy", "paired"):
            tokenizer_method = (
                HookedModelWrapper._tokenize
                if mode == "legacy"
                else paired_tokenize
            )
            w._tokenize = types.MethodType(tokenizer_method, w)

            path = ROOT / "calibration" / key / f"{mode}_logits.json"

            if path.exists():
                logits = np.array(json.loads(path.read_text()))
            else:
                logits = []

                for r in tqdm(
                    samples["positive"] + samples["negative"],
                    desc=f"{key} {mode} accuracy"
                ):
                    with torch.no_grad():
                        z = w.model(
                            **w._tokenize(text_of(r))
                        ).logits[0]
                    logits.append(z.cpu().tolist())

                save(path, logits)
                logits = np.array(logits)

            truth = np.array(
                [s["ent"]] * 100 + [s["con"]] * 100
            )
            prediction = logits.argmax(1)

            raw = get_raws(
                w,
                ROOT / "calibration" / key / mode,
                samples["positive"],
                samples["negative"],
                10
            )

            contrasts = (
                [("legacy_score", 1, 0)]
                if mode == "legacy"
                else [
                    ("legacy_score", 1, 0),
                    ("task_score", s["ent"], s["con"])
                ]
            )

            for label, ent, con in contrasts:
                stats = summarize(raw, ent, con)
                rows.append(dict(
                    model=key,
                    input=mode,
                    contrast=label,
                    accuracy=float(np.mean(prediction == truth)),
                    predicted_counts=np.bincount(
                        prediction, minlength=3
                    ).tolist(),
                    C=stats["C"],
                    VPR=stats["VPR"]
                ))

        save(ROOT / "calibration" / key / "summary.json", rows)
        display(pd.DataFrame(rows))

    finally:
        clear_model(w)


remember_cell("05_validation")

## Cell 6. Define the resumable runner

In [ ]:
def run_condition(key):
    s = SPECS[key]
    classifier = s["task"] == "classification"
    w = load_wrapper(key)

    try:
        save(
            ROOT / "checks" / f"{key}_run.json",
            check_model(key, w)
        )

        for seed in SEEDS:
            folder = ROOT / "runs" / key / f"seed_{seed}"
            examples = MANIFEST["samples"][str(seed)]

            pos = [text_of(r) for r in examples["positive"]]
            neg = [text_of(r) for r in examples["negative"]]

            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            started = time.monotonic()

            for tier in ([1, 2] if classifier else [1]):
                path = folder / f"tier{tier}.json"

                if path.exists():
                    continue

                print(
                    f"{key}, seed {seed}: Tier {tier}",
                    flush=True
                )

                with (
                    warnings.catch_warnings(record=True) as ws,
                    threadpool_limits(limits=1)
                ):
                    warnings.simplefilter("always")

                    if tier == 1:
                        result = heldout_tier1(
                            Tier1Analyzer(w),
                            pos, neg,
                            seed=seed,
                            n_perm=200
                        )
                    else:
                        result = heldout_tier2(
                            Tier2Analyzer(w),
                            pos, neg,
                            seed=seed,
                            paired=False
                        )

                score_key = (
                    "tier1_separation"
                    if tier == 1
                    else "tier2_best_accuracy"
                )
                assert np.isfinite(result[score_key]), (
                    f"Undefined Tier {tier}; stopping."
                )

                result["warnings"] = sorted(
                    set(str(v.message) for v in ws)
                )
                save(path, result)

            if classifier and not (folder / "tier3.json").exists():
                raw = get_raws(
                    w,
                    folder / "pairs",
                    examples["positive"],
                    examples["negative"],
                    10
                )
                result = summarize(raw, s["ent"], s["con"])

                # Once per model, compare the new aggregation against
                # the original analyzer using the corrected interface.
                if seed == 42:
                    a = Tier3Analyzer(w, min_gap=0.01)
                    fn = lambda z: float(
                        (z[0, s["ent"]] - z[0, s["con"]]).item()
                    )

                    try:
                        original = a.analyze_with_multiple_examples(
                            pos[:10],
                            neg[:10],
                            num_pairs=10,
                            output_fn=fn
                        )

                        assert (
                            original.num_valid_pairs
                            == result["n_valid"]
                        )

                        if result["sufficient_for_statistics"]:
                            np.testing.assert_allclose(
                                original.recovery_median,
                                result["C"],
                                rtol=1e-5,
                                atol=1e-5
                            )
                            np.testing.assert_allclose(
                                [
                                    original.layer_recovery_medians[
                                        f"layer_{l}"
                                    ]
                                    for l in range(w.num_layers)
                                ],
                                result["layer_medians"],
                                rtol=1e-5,
                                atol=1e-5
                            )
                    finally:
                        a._clear_hooks()
                        w._clear_hooks()
                        w._clear_cache()

                save(folder / "tier3.json", result)

            elapsed = time.monotonic() - started
            print(
                f"{key}, seed {seed}: saved; "
                f"{elapsed:.0f}s this pass",
                flush=True
            )

        if classifier:
            aux = MANIFEST["auxiliary"]

            raw = get_raws(
                w,
                ROOT / "auxiliary" / key / "pairs",
                aux["positive"],
                aux["negative"],
                20
            )

            ablation = [
                summarize(raw, s["ent"], s["con"], gamma)
                for gamma in PROTOCOL["gap_thresholds"]
            ]
            save(
                ROOT / "auxiliary" / key / "gap_ablation.json",
                ablation
            )

            # Correct-source donor check: first 10 auxiliary pairs,
            # at the originally specified middle block.
            _, keep, R = recoveries(
                raw[:10], s["ent"], s["con"], 0.01
            )
            layer = w.num_layers // 2
            values = R[keep, layer]

            control = dict(
                layer=layer,
                n_valid=int(keep.sum()),
                median=(
                    float(np.median(values))
                    if len(values) >= 3 else None
                ),
                iqr=(
                    float(
                        np.percentile(values, 75)
                        - np.percentile(values, 25)
                    )
                    if len(values) >= 3 else None
                ),
                description=(
                    "Correct-source check, seed42, dataset-order "
                    "positive donors; corrected replication, "
                    "not historical replay"
                )
            )
            save(
                ROOT / "auxiliary" / key / "donor_check.json",
                control
            )

    finally:
        clear_model(w)

    assert source_hashes() == SOURCE_HASHES


remember_cell("06_runner")

## Cell 7. Run classifier calibration

In [ ]:
calibration("bert_nli")
calibration("roberta_nli")

## Cell 8. Run corrected classifier measurements

In [ ]:
run_condition("bert_nli")
run_condition("roberta_nli")

## Cell 9. Run pretrained-feature comparisons

In [ ]:
run_condition("bert_pretrained")
run_condition("roberta_pretrained")

## Cell 10. Summarize corrected conditions and export

In [ ]:
rows = []
summary = []

for key, spec in SPECS.items():
    for seed in SEEDS:
        folder = ROOT / "runs" / key / f"seed_{seed}"

        t1 = json.loads((folder / "tier1.json").read_text())
        row = dict(
            model=key,
            seed=seed,
            T1=t1["tier1_separation"],
            null_p95=t1["tier1_perm_null_p95"],
            warnings_t1=len(t1["warnings"])
        )

        if spec["task"] == "classification":
            t2 = json.loads((folder / "tier2.json").read_text())
            t3 = json.loads((folder / "tier3.json").read_text())

            row.update(
                T2=t2["tier2_best_accuracy"],
                C=t3["C"],
                VPR=t3["VPR"],
                n_valid=t3["n_valid"],
                applicable=t3["applicable"],
                warnings_t2=len(t2["warnings"])
            )

        rows.append(row)

df = pd.DataFrame(rows)

for key, spec in SPECS.items():
    group = df[df.model == key]
    row = dict(
        model=key,
        seeds=len(group),
        T1=group.T1.mean()
    )

    if spec["task"] == "classification":
        profiles = [
            json.loads(
                (
                    ROOT / "runs" / key
                    / f"seed_{seed}" / "tier3.json"
                ).read_text()
            )["layer_medians"]
            for seed in SEEDS
        ]

        arr = np.array(profiles, dtype=float)
        supported = np.isfinite(arr).all(axis=1)

        profile = (
            np.median(arr, axis=0)
            if supported.all()
            else np.full(arr.shape[1], np.nan)
        )

        crossed = np.flatnonzero(profile >= 0.5)
        complete = group.C.notna().all()

        row.update(
            T2=group.T2.mean(),
            C=group.C.mean() if complete else None,
            C_q25=group.C.quantile(.25) if complete else None,
            C_q75=group.C.quantile(.75) if complete else None,
            C_defined_seeds=int(group.C.notna().sum()),
            VPR=group.VPR.mean(),
            applicable_seeds=int(group.applicable.sum()),
            profile_defined_seeds=int(supported.sum()),
            d50=(
                float(crossed[0] / (len(profile) - 1))
                if len(crossed) else None
            )
        )

        save(
            ROOT / "summary" / f"{key}_median_profile.json",
            dict(
                profile=profile,
                defined_seeds=int(supported.sum()),
                d50=row["d50"]
            )
        )

    summary.append(row)

out = pd.DataFrame(summary)
(ROOT / "summary").mkdir(exist_ok=True)

df.to_csv(ROOT / "summary" / "per_seed.csv", index=False)
out.to_csv(
    ROOT / "summary" / "corrected_conditions.csv",
    index=False
)

print("CORRECTED CONDITIONS")
display(out)

print(
    "Saved warnings:",
    int(
        df.warnings_t1.sum()
        + df.get("warnings_t2", pd.Series(dtype=float)).sum()
    )
)

# This review copy exports corrected conditions only. Historical-result
# comparisons and provisional combined-paper summaries are not inputs
# to the corrected measurements and are omitted from this code release.

assert source_hashes() == SOURCE_HASHES
remember_cell("10_summary")

bundle = Path("/content/FACI_NLI_Review_results.zip")

with zipfile.ZipFile(
    bundle, "w", zipfile.ZIP_DEFLATED
) as z:
    for path in sorted(ROOT.rglob("*")):
        if path.is_file() and not path.name.endswith(".tmp"):
            z.write(path, path.relative_to(ROOT))

files.download(str(bundle))